In [ ]:
%env REPRODUCIBLE_SEED=42
%env STRICT_DETERMINISM=1
%env CUBLAS_WORKSPACE_CONFIG=:4096:8
%env DL_NUM_WORKERS=0

In [ ]:
import os
import sys

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")

# Track if torch was already preloaded
_TORCH_WAS_PRELOADED = "torch" in sys.modules
_CUBLAS_ENV_WAS_PRESET = "CUBLAS_WORKSPACE_CONFIG" in os.environ

import json
import random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

#  REPRODUCIBILITY / SEEDING
SEED = int(os.environ.get("REPRODUCIBLE_SEED", 42))
STRICT_DETERMINISM = os.environ.get("STRICT_DETERMINISM", "1").strip().lower() not in {"0", "false", "no"}
STRICT_DETERMINISM_ENABLED = False

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def _disable_strict_determinism(reason):
    global STRICT_DETERMINISM, STRICT_DETERMINISM_ENABLED
    STRICT_DETERMINISM = False
    STRICT_DETERMINISM_ENABLED = False
    try:
        torch.use_deterministic_algorithms(False)
    except Exception:
        pass
    print("Warning: strict deterministic algorithms disabled.")
    print("         Reason:", reason)

def configure_determinism():
    global STRICT_DETERMINISM_ENABLED
    if not STRICT_DETERMINISM:
        print("Torch deterministic algorithms: DISABLED (STRICT_DETERMINISM=0)")
        return

    if torch.cuda.is_available() and _TORCH_WAS_PRELOADED and not _CUBLAS_ENV_WAS_PRESET:
        _disable_strict_determinism(
            "torch was already imported before CUBLAS_WORKSPACE_CONFIG was set."
        )
        print("         For strict CUDA determinism in Kaggle/Colab, set")
        print("         %env CUBLAS_WORKSPACE_CONFIG=:4096:8")
        print("         and restart the kernel before running this script.")
        return

    cublas_cfg = os.environ.get("CUBLAS_WORKSPACE_CONFIG", "")
    valid_cfg = {":4096:8", ":16:8"}
    if torch.cuda.is_available() and cublas_cfg not in valid_cfg:
        _disable_strict_determinism(
            f"unsupported CUBLAS_WORKSPACE_CONFIG='{cublas_cfg}'. Use :4096:8 or :16:8."
        )
        return

    try:
        torch.use_deterministic_algorithms(True)
        STRICT_DETERMINISM_ENABLED = True
        print("Torch deterministic algorithms: ENABLED")
    except Exception as e:
        _disable_strict_determinism(str(e))

set_seed(SEED)
configure_determinism()

# ===================== DataLoader deterministic helpers =====================
DL_NUM_WORKERS = int(os.environ.get("DL_NUM_WORKERS", 0))
dl_generator = torch.Generator()
dl_generator.manual_seed(SEED)

def _worker_init_fn(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)
    os.environ["PYTHONHASHSEED"] = str(worker_seed)

# CONFIGURATION 
CONDITION = "A"

SYNTH_PATH          = "/kaggle/synthetic_data_for_classification.jsonl"
DEV_TRACK_A_PATH    = "/kaggle/dev_track_a.jsonl"
TEST_TRACK_A_PATH   = "/kaggle/test_track_a.jsonl"
TEST_TRACK_B_PATH   = "/kaggle/test_track_b.jsonl"
TEST_TRACK_A_LABELS = "/kaggle/test_track_a_labels.jsonl"
TEST_TRACK_B_LABELS = "/kaggle/test_track_b_labels.jsonl"
ASPECTS_CACHE_PATH  = "/kaggle/clean_extracted_aspects_including_sample.json"

OUT_DIR = "/kaggle/working/"

MODEL_NAME = "sentence-transformers/all-roberta-large-v1"

STAGE1_EPOCHS = 1
STAGE2_EPOCHS = 2
STAGE1_LR     = 1e-5
STAGE2_LR     = 6e-6
STAGE1_BS     = 2
STAGE2_BS     = 4
PRED_BS       = 16
ENC_BS        = 8
MARGIN        = 0.3

USE_SYNTHETIC_STAGE1 = True
RUN_CV = False
CV_SPLITS = 5
RUN_FINAL_TEST = True

LABEL_SMOOTH_POS = 0.90
LABEL_SMOOTH_NEG = 0.10
FUSION_DIM = 1024

# Condition mapping (only condition G is used here, but kept for completeness)
_CONDITION_CONFIGS = {
    "A": ("Baseline", "full_text", 256, [], False, False),
    "B": ("CoA only (input)", "coa", 384, [], False, False),
    "C": ("Outcomes only (input)", "outcomes", 128, [], False, False),
    "D": ("Theme only (input)", "theme", 128, [], False, False),
    "E": ("Concat aspects", "concat", 256, [], False, False),
    "F": ("Extended context", "full_text", 256, [], False, False),
    "G": ("Aspect heads", "full_text", 512, ["coa", "outcomes", "theme"], False, False),
    "H": ("Full text + CoA/Outcomes append", "full_plus_coa_outcomes", 384, [], False, False),
    "I": ("Full text + all aspects append", "full_plus_concat", 384, [], False, False),
    "J": ("CoA head only", "full_text", 512, ["coa"], False, False),
    "K": ("Outcomes head only", "full_text", 512, ["outcomes"], False, False),
    "L": ("Theme head only", "full_text", 512, ["theme"], False, False),
    "M": ("CoA+Outcomes heads", "full_text", 512, ["coa", "outcomes"], False, False),
    "N": ("CoA+Outcomes heads + appended aspects", "full_plus_coa_outcomes", 384, ["coa", "outcomes"], False, False),
    "O": ("Improved multi-view (512 + soft labels)", "full_text", 512, ["coa", "outcomes", "theme"], True, True),
    "P": ("Multi-view aspect forwards", "full_text", 256, ["coa", "outcomes", "theme"], False, True),
}

_CONDITION_NOTES = {
    "G": "Full-text encoder with three auxiliary aspect heads.",
}

assert CONDITION in _CONDITION_CONFIGS
_cname, INPUT_MODE, MAX_LEN, ASPECTS, USE_SOFT_LABELS, USE_MULTIVIEW = _CONDITION_CONFIGS[CONDITION]
_cnote = _CONDITION_NOTES.get(CONDITION, "")
USE_ASPECT_HEADS = bool(ASPECTS) and not USE_MULTIVIEW
ACTIVE_ASPECTS = list(ASPECTS)

if RUN_FINAL_TEST and not RUN_CV:
    suffix = "full_train"
elif RUN_CV:
    suffix = f"cv_{CV_SPLITS}folds"
else:
    suffix = "results"

RUN_PREFIX = f"Condition_{CONDITION}_{MAX_LEN}_{suffix}"

OUT_TRACK_A = os.path.join(OUT_DIR, f"{RUN_PREFIX}_track_a.jsonl")
OUT_TRACK_B = os.path.join(OUT_DIR, f"{RUN_PREFIX}_track_b.npy")
METRICS_JSON = os.path.join(OUT_DIR, f"{RUN_PREFIX}_eval_metrics.json")
CV_RESULTS_JSON = os.path.join(OUT_DIR, f"{RUN_PREFIX}_cv_results.json")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ===================== ASPECT CACHE =====================
def _norm_key(text):
    return " ".join(str(text).split())

def _read_json_or_jsonl(path):
    p = Path(path)
    if not p.exists():
        return None
    text = p.read_text(encoding="utf-8").strip()
    if not text:
        return None
    if text[0] in "[{":
        try:
            return json.loads(text)
        except Exception:
            pass
    rows = []
    with open(p, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def load_aspects_cache(path):
    raw = _read_json_or_jsonl(path)
    if raw is None:
        return {}
    cache = {}
    def add_record(story_text, payload):
        if not story_text:
            return
        cache[_norm_key(story_text)] = {
            "coa": payload.get("coa") or payload.get("course_of_action") or payload.get("plot") or "",
            "outcomes": payload.get("outcomes") or payload.get("outcome") or payload.get("ending") or "",
            "theme": payload.get("theme") or payload.get("abstract_theme") or "",
            "title": payload.get("title", ""),
        }
    if isinstance(raw, list):
        for obj in raw:
            if isinstance(obj, dict):
                add_record(obj.get("text") or obj.get("raw_text") or obj.get("story"), obj)
    elif isinstance(raw, dict):
        if "records" in raw and isinstance(raw["records"], list):
            for obj in raw["records"]:
                if isinstance(obj, dict):
                    add_record(obj.get("text") or obj.get("raw_text") or obj.get("story"), obj)
        else:
            for k, v in raw.items():
                if isinstance(v, dict):
                    add_record(k, v)
    return cache

ASPECTS_CACHE = load_aspects_cache(ASPECTS_CACHE_PATH)

def _nonempty_parts(parts):
    return [p for p in parts if str(p).strip()]

def get_story_input(text, mode):
    if mode == "full_text":
        return text
    entry = ASPECTS_CACHE.get(_norm_key(text), {})
    coa = entry.get("coa", text) or text
    outcomes = entry.get("outcomes", text) or text
    theme = entry.get("theme", text) or text

    if mode == "coa":
        return coa
    if mode == "outcomes":
        return outcomes
    if mode == "theme":
        return theme
    if mode == "concat":
        parts = _nonempty_parts([entry.get("coa", ""), entry.get("outcomes", ""), entry.get("theme", "")])
        return " [SEP] ".join(parts) if parts else text
    if mode == "full_plus_coa_outcomes":
        parts = _nonempty_parts([text, entry.get("coa", ""), entry.get("outcomes", "")])
        return " [SEP] ".join(parts)
    if mode == "full_plus_concat":
        parts = _nonempty_parts([text, entry.get("coa", ""), entry.get("outcomes", ""), entry.get("theme", "")])
        return " [SEP] ".join(parts)
    return text

def get_story_views(text):
    entry = ASPECTS_CACHE.get(_norm_key(text), {})
    return {
        "full_text": text,
        "coa": entry.get("coa", text) or text,
        "outcomes": entry.get("outcomes", text) or text,
        "theme": entry.get("theme", text) or text,
    }

def batch_to_views(text_list):
    views = [get_story_views(t) for t in text_list]
    return {
        "full_text": [v["full_text"] for v in views],
        "coa": [v["coa"] for v in views],
        "outcomes": [v["outcomes"] for v in views],
        "theme": [v["theme"] for v in views],
    }

# ===================== MODEL (with gradient checkpointing) =====================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# Load a single encoder with gradient checkpointing enabled
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
if DEVICE == "cuda":
    encoder.gradient_checkpointing_enable()
    print("Gradient checkpointing: ENABLED")
else:
    print("Gradient checkpointing: DISABLED (CPU)")

HIDDEN_DIM = encoder.config.hidden_size
PROJ_DIM = 256

def tokenize(texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)

def mean_pool(model_output, attention_mask):
    token_emb = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).float()
    summed = (token_emb * mask).sum(1)
    counts = mask.sum(1).clamp(min=1e-9)
    return F.normalize(summed / counts, p=2, dim=1)

class NarrativeModel(nn.Module):
    def __init__(self, encoder, hidden_dim, proj_dim, aspect_keys):
        super().__init__()
        self.encoder = encoder
        self.aspect_keys = list(aspect_keys) if aspect_keys else []
        for key in self.aspect_keys:
            if key == "coa":
                self.head_coa = nn.Linear(hidden_dim, proj_dim)
            elif key == "outcomes":
                self.head_outcomes = nn.Linear(hidden_dim, proj_dim)
            elif key == "theme":
                self.head_theme = nn.Linear(hidden_dim, proj_dim)

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return mean_pool(out, attention_mask)

    def forward(self, input_ids, attention_mask):
        emb = self.encode(input_ids, attention_mask)
        if not self.aspect_keys:
            return emb
        out = {"global": emb}
        for key in self.aspect_keys:
            if key == "coa":
                out["coa"] = F.normalize(self.head_coa(emb), p=2, dim=1)
            elif key == "outcomes":
                out["outcomes"] = F.normalize(self.head_outcomes(emb), p=2, dim=1)
            elif key == "theme":
                out["theme"] = F.normalize(self.head_theme(emb), p=2, dim=1)
        return out

class MultiViewNarrativeModel(nn.Module):
    def __init__(self, encoder, hidden_dim, fusion_dim=1024):
        super().__init__()
        self.encoder = encoder
        self.fusion = nn.Linear(hidden_dim * 4, fusion_dim)

    def encode_texts(self, texts):
        enc = tokenize(texts)
        out = self.encoder(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
        return mean_pool(out, enc["attention_mask"])

    def forward_views(self, full_texts, coa_texts, outcomes_texts, theme_texts):
        z_full = self.encode_texts(full_texts)
        z_coa = self.encode_texts(coa_texts)
        z_out = self.encode_texts(outcomes_texts)
        z_theme = self.encode_texts(theme_texts)
        fused = torch.cat([z_full, z_coa, z_out, z_theme], dim=1)
        fused = F.normalize(self.fusion(fused), p=2, dim=1)
        return {"full": z_full, "coa": z_coa, "outcomes": z_out, "theme": z_theme, "fused": fused}

def build_model(encoder_with_checkpointing):
    if USE_MULTIVIEW:
        return MultiViewNarrativeModel(encoder_with_checkpointing, HIDDEN_DIM, fusion_dim=FUSION_DIM).to(DEVICE)
    return NarrativeModel(encoder_with_checkpointing, HIDDEN_DIM, PROJ_DIM, ACTIVE_ASPECTS).to(DEVICE)

model = build_model(encoder)

# ===================== DATASETS =====================
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

class SyntheticTripletDataset(Dataset):
    def __init__(self, path):
        self.data = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                anchor = obj.get("anchor_text", "")
                ta = obj.get("text_a", "")
                tb = obj.get("text_b", "")
                closer = obj.get("text_a_is_closer")
                if not anchor or not ta or not tb:
                    continue
                if len(anchor) < 20 or len(ta) < 20 or len(tb) < 20:
                    continue
                pos, neg = (ta, tb) if closer else (tb, ta)
                if USE_MULTIVIEW:
                    self.data.append((anchor, pos, neg))
                else:
                    self.data.append((get_story_input(anchor, INPUT_MODE),
                                      get_story_input(pos, INPUT_MODE),
                                      get_story_input(neg, INPUT_MODE)))
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

class TrackARowsDataset(Dataset):
    def __init__(self, rows):
        self.rows_eval = list(rows)
        self.data = []
        for obj in rows:
            if USE_MULTIVIEW:
                self.data.append((obj["anchor_text"], obj["text_a"], obj["text_b"], bool(obj["text_a_is_closer"])))
            else:
                self.data.append((get_story_input(obj["anchor_text"], INPUT_MODE),
                                  get_story_input(obj["text_a"], INPUT_MODE),
                                  get_story_input(obj["text_b"], INPUT_MODE),
                                  bool(obj["text_a_is_closer"])))
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

class TrackATestDataset(Dataset):
    def __init__(self, path):
        self.data = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                if USE_MULTIVIEW:
                    self.data.append((obj["anchor_text"], obj["text_a"], obj["text_b"]))
                else:
                    self.data.append((get_story_input(obj["anchor_text"], INPUT_MODE),
                                      get_story_input(obj["text_a"], INPUT_MODE),
                                      get_story_input(obj["text_b"], INPUT_MODE)))
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

# ===================== LOSSES & HELPERS =====================
def _get_global_emb(output):
    return output["global"] if isinstance(output, dict) else output

def _triplet_loss(a, p, n, margin=MARGIN):
    return F.triplet_margin_loss(a, p, n, margin=margin)

def _rank_loss(a, p, n):
    return F.relu(F.cosine_similarity(a, n).mean() - F.cosine_similarity(a, p).mean() + 0.2)

def _aspect_triplet_loss(out_a, out_p, out_n):
    if not ACTIVE_ASPECTS:
        return torch.tensor(0.0, device=DEVICE)
    return sum(_triplet_loss(out_a[h], out_p[h], out_n[h]) for h in ACTIVE_ASPECTS) / float(len(ACTIVE_ASPECTS))

def _multiview_stage1_loss(out_a, out_p, out_n):
    return (0.40 * _triplet_loss(out_a["fused"], out_p["fused"], out_n["fused"]) +
            0.15 * _triplet_loss(out_a["full"], out_p["full"], out_n["full"]) +
            0.15 * _triplet_loss(out_a["coa"], out_p["coa"], out_n["coa"]) +
            0.15 * _triplet_loss(out_a["outcomes"], out_p["outcomes"], out_n["outcomes"]) +
            0.15 * _triplet_loss(out_a["theme"], out_p["theme"], out_n["theme"]))

def _multiview_score(out_anchor, out_candidate):
    return (0.35 * F.cosine_similarity(out_anchor["full"], out_candidate["full"]) +
            0.30 * F.cosine_similarity(out_anchor["coa"], out_candidate["coa"]) +
            0.20 * F.cosine_similarity(out_anchor["outcomes"], out_candidate["outcomes"]) +
            0.15 * F.cosine_similarity(out_anchor["theme"], out_candidate["theme"]))

# ===================== TRAINING & EVALUATION =====================
def train_stage1(model, dataset, epochs=STAGE1_EPOCHS):
    if len(dataset) == 0:
        return model
    loader = DataLoader(dataset, batch_size=STAGE1_BS, shuffle=True,
                        num_workers=DL_NUM_WORKERS, worker_init_fn=_worker_init_fn, generator=dl_generator)
    optimizer = torch.optim.AdamW(model.parameters(), lr=STAGE1_LR)
    model.train()
    for ep in range(epochs):
        loop = tqdm(loader, desc=f"Stage1 Epoch {ep+1}")
        running = 0.0
        for anchors, positives, negatives in loop:
            if USE_MULTIVIEW:
                a_views = batch_to_views(list(anchors))
                p_views = batch_to_views(list(positives))
                n_views = batch_to_views(list(negatives))
                out_a = model.forward_views(a_views["full_text"], a_views["coa"], a_views["outcomes"], a_views["theme"])
                out_p = model.forward_views(p_views["full_text"], p_views["coa"], p_views["outcomes"], p_views["theme"])
                out_n = model.forward_views(n_views["full_text"], n_views["coa"], n_views["outcomes"], n_views["theme"])
                loss = _multiview_stage1_loss(out_a, out_p, out_n)
            else:
                texts = list(anchors) + list(positives) + list(negatives)
                enc = tokenize(texts)
                out = model(enc["input_ids"], enc["attention_mask"])
                B = len(anchors)
                if USE_ASPECT_HEADS:
                    ga, gp, gn = out["global"][:B], out["global"][B:2*B], out["global"][2*B:]
                    a_heads = {k: out[k][:B] for k in ACTIVE_ASPECTS}
                    p_heads = {k: out[k][B:2*B] for k in ACTIVE_ASPECTS}
                    n_heads = {k: out[k][2*B:] for k in ACTIVE_ASPECTS}
                    loss = (0.7 * _triplet_loss(ga, gp, gn) +
                            0.3 * _rank_loss(ga, gp, gn) +
                            0.3 * _aspect_triplet_loss(a_heads, p_heads, n_heads))
                else:
                    a, p, n = out[:B], out[B:2*B], out[2*B:]
                    loss = 0.7 * _triplet_loss(a, p, n) + 0.3 * _rank_loss(a, p, n)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += loss.item()
            loop.set_postfix(loss=f"{running / (loop.n + 1):.4f}")
    model.eval()
    return model

def _macro_f1_from_binary(y_true, y_pred):
    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)
    tp = int(( y_pred &  y_true).sum())
    tn = int((~y_pred & ~y_true).sum())
    fp = int(( y_pred & ~y_true).sum())
    fn = int((~y_pred &  y_true).sum())
    f1_pos = 2 * tp / (2 * tp + fp + fn + 1e-9)
    f1_neg = 2 * tn / (2 * tn + fn + fp + 1e-9)
    return (f1_pos + f1_neg) / 2.0, tp, tn, fp, fn

def evaluate_trackA_rows(model, rows):
    ds = TrackARowsDataset(rows)
    loader = DataLoader(ds, batch_size=PRED_BS, shuffle=False,
                        num_workers=DL_NUM_WORKERS, worker_init_fn=_worker_init_fn)
    y_true, y_pred = [], []
    model.eval()
    with torch.no_grad():
        for anchors, ta_list, tb_list, labels in loader:
            if USE_MULTIVIEW:
                a_views = batch_to_views(list(anchors))
                ta_views = batch_to_views(list(ta_list))
                tb_views = batch_to_views(list(tb_list))
                out_a = model.forward_views(a_views["full_text"], a_views["coa"], a_views["outcomes"], a_views["theme"])
                out_ta = model.forward_views(ta_views["full_text"], ta_views["coa"], ta_views["outcomes"], ta_views["theme"])
                out_tb = model.forward_views(tb_views["full_text"], tb_views["coa"], tb_views["outcomes"], tb_views["theme"])
                score = _multiview_score(out_a, out_ta) - _multiview_score(out_a, out_tb)
                pred = (score > 0).cpu().tolist()
            else:
                texts = list(anchors) + list(ta_list) + list(tb_list)
                enc = tokenize(texts)
                out = model(enc["input_ids"], enc["attention_mask"])
                B = len(anchors)
                g = _get_global_emb(out)
                a, ta, tb = g[:B], g[B:2*B], g[2*B:]
                pred = (F.cosine_similarity(a, ta) > F.cosine_similarity(a, tb)).cpu().tolist()
            y_pred.extend([bool(x) for x in pred])
            y_true.extend([bool(x) for x in labels])
    acc = accuracy_score(y_true, y_pred)
    macro_f1, tp, tn, fp, fn = _macro_f1_from_binary(y_true, y_pred)
    return {"accuracy": float(acc), "macro_f1": float(macro_f1), "tp": tp, "tn": tn, "fp": fp, "fn": fn}

def train_stage2(model, dataset, epochs=STAGE2_EPOCHS):
    loader = DataLoader(dataset, batch_size=STAGE2_BS, shuffle=True,
                        num_workers=DL_NUM_WORKERS, worker_init_fn=_worker_init_fn, generator=dl_generator)
    optimizer = torch.optim.AdamW(model.parameters(), lr=STAGE2_LR)
    criterion = nn.BCEWithLogitsLoss()
    model.train()
    for ep in range(epochs):
        loop = tqdm(loader, desc=f"Stage2 Epoch {ep+1}")
        running = 0.0
        for anchors, ta_list, tb_list, closer_list in loop:
            y = torch.tensor(closer_list, dtype=torch.float32, device=DEVICE)
            if USE_SOFT_LABELS:
                y = y * LABEL_SMOOTH_POS + (1.0 - y) * LABEL_SMOOTH_NEG
            if USE_MULTIVIEW:
                a_views = batch_to_views(list(anchors))
                ta_views = batch_to_views(list(ta_list))
                tb_views = batch_to_views(list(tb_list))
                out_a = model.forward_views(a_views["full_text"], a_views["coa"], a_views["outcomes"], a_views["theme"])
                out_ta = model.forward_views(ta_views["full_text"], ta_views["coa"], ta_views["outcomes"], ta_views["theme"])
                out_tb = model.forward_views(tb_views["full_text"], tb_views["coa"], tb_views["outcomes"], tb_views["theme"])
                score = _multiview_score(out_a, out_ta) - _multiview_score(out_a, out_tb)
            else:
                texts = list(anchors) + list(ta_list) + list(tb_list)
                enc = tokenize(texts)
                out = model(enc["input_ids"], enc["attention_mask"])
                B = len(anchors)
                g = _get_global_emb(out)
                a, ta, tb = g[:B], g[B:2*B], g[2*B:]
                score = F.cosine_similarity(a, ta) - F.cosine_similarity(a, tb)
            loss = criterion(score, y)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += loss.item()
            loop.set_postfix(loss=f"{running / (loop.n + 1):.4f}")

        epoch_eval = evaluate_trackA_rows(model, dataset.rows_eval)
        print(
            f"Stage2 Epoch {ep+1} summary | "
            f"loss={running / max(len(loader), 1):.4f} | "
            f"acc={epoch_eval['accuracy']*100:.2f}% | "
            f"macro_f1={epoch_eval['macro_f1']*100:.2f}%"
        )
    model.eval()
    return model

def evaluate_trackB_style_rows(model, rows):
    unique_texts, seen = [], set()
    for obj in rows:
        for key in ["anchor_text", "text_a", "text_b"]:
            txt = obj[key].strip()
            if txt not in seen:
                seen.add(txt)
                unique_texts.append(txt)
    embs = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(unique_texts), ENC_BS):
            batch_raw = unique_texts[i:i+ENC_BS]
            if USE_MULTIVIEW:
                views = batch_to_views(batch_raw)
                out = model.forward_views(views["full_text"], views["coa"], views["outcomes"], views["theme"])
                emb = out["fused"]
            else:
                batch_inp = [get_story_input(t, INPUT_MODE) for t in batch_raw]
                enc = tokenize(batch_inp)
                out = model(enc["input_ids"], enc["attention_mask"])
                emb = torch.cat([out["global"]] + [out[k] for k in ACTIVE_ASPECTS], dim=1) if USE_ASPECT_HEADS else _get_global_emb(out)
                emb = F.normalize(emb, p=2, dim=1) if USE_ASPECT_HEADS else emb
            embs.append(emb.cpu().numpy().astype(np.float32))
    X = np.vstack(embs)
    X = X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-12, None)
    lookup = {t: X[i] for i, t in enumerate(unique_texts)}
    y_true, y_pred = [], []
    for obj in rows:
        a = lookup[obj["anchor_text"].strip()]
        ta = lookup[obj["text_a"].strip()]
        tb = lookup[obj["text_b"].strip()]
        y_pred.append(float(np.dot(a, ta)) > float(np.dot(a, tb)))
        y_true.append(bool(obj["text_a_is_closer"]))
    acc = accuracy_score(y_true, y_pred)
    macro_f1, tp, tn, fp, fn = _macro_f1_from_binary(y_true, y_pred)
    return {"accuracy": float(acc), "macro_f1": float(macro_f1), "tp": tp, "tn": tn, "fp": fp, "fn": fn}

def run_cross_validation(initial_model_state, dev_path, n_splits=5):
    rows = load_jsonl(dev_path)
    labels = [int(bool(r["text_a_is_closer"])) for r in rows]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_results = []
    for fold_id, (train_idx, val_idx) in enumerate(skf.split(rows, labels), start=1):
        train_rows = [rows[i] for i in train_idx]
        val_rows = [rows[i] for i in val_idx]
        fold_model = build_model(encoder)   # reuse the same encoder (already checkpointed)
        fold_model.load_state_dict(initial_model_state)
        fold_model = train_stage2(fold_model, TrackARowsDataset(train_rows), epochs=STAGE2_EPOCHS)
        res_a = evaluate_trackA_rows(fold_model, val_rows)
        res_b = evaluate_trackB_style_rows(fold_model, val_rows)
        fold_results.append({"fold": fold_id, "track_a_accuracy": res_a["accuracy"], "track_a_macro_f1": res_a["macro_f1"],
                             "track_b_style_accuracy": res_b["accuracy"], "track_b_style_macro_f1": res_b["macro_f1"],
                             "n_train": len(train_rows), "n_val": len(val_rows)})
    summary = {"condition": CONDITION, "name": _cname, "condition_note": _cnote, "n_splits": n_splits,
               "track_a_cv_mean": float(np.mean([r["track_a_accuracy"] for r in fold_results])),
               "track_a_cv_std": float(np.std([r["track_a_accuracy"] for r in fold_results])),
               "track_b_style_cv_mean": float(np.mean([r["track_b_style_accuracy"] for r in fold_results])),
               "track_b_style_cv_std": float(np.std([r["track_b_style_accuracy"] for r in fold_results])),
               "folds": fold_results}
    with open(CV_RESULTS_JSON, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    return summary

def write_trackA_predictions(model, test_path, out_path):
    ds = TrackATestDataset(test_path)
    loader = DataLoader(ds, batch_size=PRED_BS, shuffle=False,
                        num_workers=DL_NUM_WORKERS, worker_init_fn=_worker_init_fn)
    preds = []
    model.eval()
    with torch.no_grad():
        for anchors, ta_list, tb_list in loader:
            if USE_MULTIVIEW:
                a_views = batch_to_views(list(anchors))
                ta_views = batch_to_views(list(ta_list))
                tb_views = batch_to_views(list(tb_list))
                out_a = model.forward_views(a_views["full_text"], a_views["coa"], a_views["outcomes"], a_views["theme"])
                out_ta = model.forward_views(ta_views["full_text"], ta_views["coa"], ta_views["outcomes"], ta_views["theme"])
                out_tb = model.forward_views(tb_views["full_text"], tb_views["coa"], tb_views["outcomes"], tb_views["theme"])
                score = _multiview_score(out_a, out_ta) - _multiview_score(out_a, out_tb)
                batch_preds = (score > 0).cpu().tolist()
            else:
                texts = list(anchors) + list(ta_list) + list(tb_list)
                enc = tokenize(texts)
                out = model(enc["input_ids"], enc["attention_mask"])
                B = len(anchors)
                g = _get_global_emb(out)
                a, ta, tb = g[:B], g[B:2*B], g[2*B:]
                batch_preds = (F.cosine_similarity(a, ta) > F.cosine_similarity(a, tb)).cpu().tolist()
            preds.extend(batch_preds)
    with open(test_path, encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line, p in zip(fin, preds):
            obj = json.loads(line)
            obj["text_a_is_closer"] = bool(p)
            fout.write(json.dumps(obj) + "\n")

def build_trackB_embeddings(model, in_path, out_npy):
    texts = []
    with open(in_path, encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            texts.append(obj["text"])
    embs = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), ENC_BS):
            batch_raw = texts[i:i+ENC_BS]
            if USE_MULTIVIEW:
                views = batch_to_views(batch_raw)
                out = model.forward_views(views["full_text"], views["coa"], views["outcomes"], views["theme"])
                emb = out["fused"]
            else:
                batch_inp = [get_story_input(t, INPUT_MODE) for t in batch_raw]
                enc = tokenize(batch_inp)
                out = model(enc["input_ids"], enc["attention_mask"])
                emb = torch.cat([out["global"]] + [out[k] for k in ACTIVE_ASPECTS], dim=1) if USE_ASPECT_HEADS else _get_global_emb(out)
                emb = F.normalize(emb, p=2, dim=1) if USE_ASPECT_HEADS else emb
            embs.append(emb.cpu().numpy().astype(np.float32))
    X = np.vstack(embs).astype(np.float32)
    X = X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-12, None)
    np.save(out_npy, X)

def evaluate_trackA(predictions_path, gold_labels_path):
    pred = pd.read_json(predictions_path, lines=True)
    gold = pd.read_json(gold_labels_path, lines=True)
    y_pred = pred["text_a_is_closer"].astype(bool).tolist()
    y_true = gold["text_a_is_closer"].astype(bool).tolist()
    acc = accuracy_score(y_true, y_pred)
    macro_f1, tp, tn, fp, fn = _macro_f1_from_binary(y_true, y_pred)
    return {"accuracy": float(acc), "macro_f1": float(macro_f1), "tp": tp, "tn": tn, "fp": fp, "fn": fn}

def evaluate_trackB(track_b_input_path, embeddings_npy, gold_labels_path):
    pred = pd.read_json(track_b_input_path, lines=True)
    embs = np.load(embeddings_npy, allow_pickle=False)
    normed = embs / np.clip(np.linalg.norm(embs, axis=1, keepdims=True), 1e-12, None)
    lookup = dict(zip(pred["text"], normed))
    df = pd.read_json(gold_labels_path, lines=True)
    for col in ["anchor_text", "text_a", "text_b"]:
        df[col] = df[col].astype(str).str.strip()
    df["anchor_embedding"] = df["anchor_text"].map(lookup)
    df["a_embedding"] = df["text_a"].map(lookup)
    df["b_embedding"] = df["text_b"].map(lookup)
    missing = int((df["anchor_embedding"].isna() | df["a_embedding"].isna() | df["b_embedding"].isna()).sum())
    if missing:
        df = df[~(df["anchor_embedding"].isna() | df["a_embedding"].isna() | df["b_embedding"].isna())].copy()
    y_true = df["text_a_is_closer"].astype(bool).tolist()
    y_pred = [float(r["anchor_embedding"].dot(r["a_embedding"])) > float(r["anchor_embedding"].dot(r["b_embedding"])) for _, r in df.iterrows()]
    acc = accuracy_score(y_true, y_pred)
    macro_f1, tp, tn, fp, fn = _macro_f1_from_binary(y_true, y_pred)
    return {"accuracy": float(acc), "macro_f1": float(macro_f1), "tp": tp, "tn": tn, "fp": fp, "fn": fn, "missing": missing, "embedding_dim": int(normed.shape[1])}

def print_summary(res_a, res_b, cv_summary=None):
    payload = {"condition": CONDITION, "name": _cname, "condition_note": _cnote,
               "input_mode": INPUT_MODE, "max_len": MAX_LEN, "use_aspect_heads": USE_ASPECT_HEADS,
               "use_multiview": USE_MULTIVIEW, "active_aspects": ACTIVE_ASPECTS,
               "use_soft_labels": USE_SOFT_LABELS, "track_a_test": res_a, "track_b_test": res_b, "cv": cv_summary}
    print("\n" + "="*55)
    print(f"  FINAL SUMMARY — Condition {CONDITION}: {_cname}")
    print("="*55)
    print(f"  Track A accuracy : {res_a['accuracy']*100:.2f}%")
    print(f"  Track A macro-F1 : {res_a['macro_f1']*100:.2f}%")
    print(f"  Track B accuracy : {res_b['accuracy']*100:.2f}%")
    print(f"  Track B macro-F1 : {res_b['macro_f1']*100:.2f}%")
    print("="*55)
    with open(METRICS_JSON, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

# ===================== RUN =====================
print(f"Starting run for condition {CONDITION}: {_cname}")
print(f"Condition note: {_cnote}")

if USE_SYNTHETIC_STAGE1:
    if DEVICE != "cpu":
        synth_ds = SyntheticTripletDataset(SYNTH_PATH)
        model = train_stage1(model, synth_ds, epochs=STAGE1_EPOCHS)

initial_model_state = deepcopy(model.state_dict())

cv_summary = run_cross_validation(initial_model_state, DEV_TRACK_A_PATH, n_splits=CV_SPLITS) if RUN_CV else None

if RUN_FINAL_TEST:
    model.load_state_dict(initial_model_state)
    full_dev_rows = load_jsonl(DEV_TRACK_A_PATH)
    model = train_stage2(model, TrackARowsDataset(full_dev_rows), epochs=STAGE2_EPOCHS)
    write_trackA_predictions(model, TEST_TRACK_A_PATH, OUT_TRACK_A)
    build_trackB_embeddings(model, TEST_TRACK_B_PATH, OUT_TRACK_B)
    res_a = evaluate_trackA(OUT_TRACK_A, TEST_TRACK_A_LABELS)
    res_b = evaluate_trackB(TEST_TRACK_B_PATH, OUT_TRACK_B, TEST_TRACK_B_LABELS)
    print_summary(res_a, res_b, cv_summary=cv_summary)